# 00. Baseline (결측치 0909 ver + 파생변수 17개 + 인코딩 확정본)

팀에서 확정한 결측치/중복 처리, 파생변수 17개, 인코딩 방식을 모두 반영한 기준 베이스라인입니다.

In [27]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42  # 그라운드룰 1: 항상 42로 고정

## 1. Data Load

In [28]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

print('train:', train.shape, '/ test:', test.shape)

train: (3000, 18) / test: (3000, 17)


## 2. 결측치 및 중복행 처리 (0909 ver, 팀 확정본)

- `mean_working`: 은퇴/미취업 상태로 추정 → `0`으로 대체
- `medical_history`, `family_medical_history`: 병력 없음으로 추정 → `'None'` 카테고리
- `edu_level`: 미응답으로 추정 → `'Unknown'` 카테고리
- `bone_density`: 원본 수치 그대로 유지 (합성 데이터 노이즈로 추정, train/test 동일 패턴)
- train 완전 중복 행(ID 제외 6쌍): K-Fold 리키지 방지를 위해 제거

In [29]:
# 처리 전 결측치 개수 확인 (컬럼별)
print('train 결측치:')
print(train.isnull().sum()[train.isnull().sum() > 0])
print()
print('test 결측치:')
print(test.isnull().sum()[test.isnull().sum() > 0])
print()
print('중복 행 개수(ID 제외):', train.duplicated(subset=[c for c in train.columns if c != 'ID']).sum())

train 결측치:
medical_history           1289
family_medical_history    1486
edu_level                  607
mean_working              1032
dtype: int64

test 결측치:
medical_history           1309
family_medical_history    1416
edu_level                  647
mean_working              1008
dtype: int64

중복 행 개수(ID 제외): 6


In [30]:
# 중복 행 제거 (ID 제외 기준) - train에만 적용, test는 제거하지 않음
train = train.drop_duplicates(
    subset=[col for col in train.columns if col not in ['ID']]
).reset_index(drop=True)

# 근로시간 결측치: 0 처리
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

# 범주형 결측치: 독립 범주 신설
for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

# bone_density: 원본 수치 그대로 유지 (처리 없음)

print('결측치 처리 후 남은 결측 개수 - train:', train.isnull().sum().sum(), '/ test:', test.isnull().sum().sum())
print('중복 제거 후 train shape:', train.shape)

결측치 처리 후 남은 결측 개수 - train: 0 / test: 0
중복 제거 후 train shape: (2994, 18)


## 3. 파생변수 생성 (0909 ver, 팀 확정본 17개)

**중요**: 반드시 2단계(결측치 fillna)가 끝난 뒤, 4단계(인코딩) 이전에 실행해야 합니다.
`medical_history != 'None'` 같은 로직이 문자열 상태의 원본 카테고리에 의존하기 때문에,
인코딩 이후에 실행하면 값이 전부 깨집니다.

In [31]:
def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)

    # 1. 과로 및 생활 리듬
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)

    # 2. 질환 및 유전력
    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)

    # 3. 심혈관 및 신체
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)

    # 4. 대사 및 노화
    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)

    return data

train = add_features(train)
test = add_features(test)

print('파생변수 추가 후 train shape:', train.shape)

파생변수 추가 후 train shape: (2994, 35)


In [32]:
# 새로 추가된 파생변수 컬럼 확인
original_cols = set(pd.read_csv('../data/train.csv').columns)
new_cols = [c for c in train.columns if c not in original_cols]
print('추가된 파생변수 개수:', len(new_cols))
print(new_cols)
train[new_cols].head()

추가된 파생변수 개수: 17
['is_overworking', 'work_sleep_risk', 'oversleep_low_activity', 'working_age_ratio', 'activity_sleep_mismatch', 'smoker_with_disease', 'age_disease_interaction', 'has_medical_history', 'has_family_history', 'total_disease_burden', 'genetic_risk_match', 'bmi', 'pulse_pressure', 'map', 'is_hypertension', 'is_low_bone_density', 'glucose_chol_ratio']


,is_overworking,work_sleep_risk,oversleep_low_activity,working_age_ratio,activity_sleep_mismatch,smoker_with_disease,age_disease_interaction,has_medical_history,has_family_history,total_disease_burden,genetic_risk_match,bmi,pulse_pressure,map,is_hypertension,is_low_bone_density,glucose_chol_ratio
0,0,0,0,0.0000,0,0,72,1,1,2,0,22.420321,65,121.666667,1,0,0.510433
1,0,0,0,0.0000,0,0,0,0,1,1,0,23.985250,67,133.333333,1,0,0.568719
2,0,0,0,0.1875,0,0,0,0,0,0,0,27.009817,39,108.000000,1,0,0.626417
3,0,0,0,0.0000,0,0,69,1,0,1,0,19.884564,66,114.000000,1,0,0.660730
4,0,0,0,0.0000,0,0,81,1,1,2,1,26.391876,55,134.333333,1,0,0.503542


## 4. 인코딩 (Ordinal + Label 혼합, 팀 확정본)

- `activity`: 순서형(Ordinal) — light=0, moderate=1, intense=2
- `edu_level`: 순서형(Ordinal) — Unknown=0, high school diploma=1, bachelors degree=2, graduate degree=3
- 나머지 5개(`gender`, `smoke_status`, `medical_history`, `family_medical_history`, `sleep_pattern`): 기본 LabelEncoder

In [33]:
activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}

train['activity'] = train['activity'].map(activity_map)
test['activity'] = test['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
test['edu_level'] = test['edu_level'].map(edu_map)

nominal_cols = ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern']

for feature in nominal_cols:
    le = LabelEncoder()
    le = le.fit(train[feature])
    train[feature] = le.transform(train[feature])

    unseen = [label for label in np.unique(test[feature]) if label not in le.classes_]
    if unseen:
        le.classes_ = np.append(le.classes_, unseen)
    test[feature] = le.transform(test[feature])

x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score']
x_test = test.drop('ID', axis=1)

print('x_train:', x_train.shape, '/ x_test:', x_test.shape)

x_train: (2994, 33) / x_test: (3000, 33)


In [34]:
# 인코딩 결과 확인 (문자열 컬럼이 다 숫자로 바뀌었는지)
print('object(문자열) 타입으로 남은 컬럼:', list(x_train.select_dtypes(include='object').columns))
x_train.head()

object(문자열) 타입으로 남은 컬럼: []


,gender,age,height,weight,cholesterol,systolic_blood_pressure,diastolic_blood_pressure,glucose,bone_density,activity,...,has_medical_history,has_family_history,total_disease_burden,genetic_risk_match,bmi,pulse_pressure,map,is_hypertension,is_low_bone_density,glucose_chol_ratio
0,0,72,161.49,58.47,279.84,165,100,143.35,0.87,1,...,1,1,2,0,22.420321,65,121.666667,1,0,0.510433
1,1,88,179.87,77.60,257.37,178,111,146.94,0.07,1,...,0,1,1,0,23.985250,67,133.333333,1,0,0.568719
2,1,47,182.47,89.93,226.66,134,95,142.61,1.18,0,...,0,0,0,0,27.009817,39,108.000000,1,0,0.626417
3,1,69,185.78,68.63,206.74,158,92,137.26,0.48,2,...,1,0,1,0,19.884564,66,114.000000,1,0,0.660730
4,0,81,164.63,71.53,255.92,171,116,129.37,0.34,1,...,1,1,2,1,26.391876,55,134.333333,1,0,0.503542


## 5. 5-Fold CV로 기준 점수(#00) 측정

그라운드룰 3(로컬 CV 우선주의)에 따라, 이 점수를 팀의 현재 최고 기록 기준선으로 삼습니다.

In [35]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
val_maes = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(x_train), 1):
    X_tr, X_val = x_train.iloc[tr_idx], x_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = LGBMRegressor(random_state=RANDOM_STATE)
    model.fit(X_tr, y_tr)

    val_pred = model.predict(X_val)
    val_mae = mean_absolute_error(y_val, val_pred)
    val_maes.append(val_mae)
    print(f'fold {fold}: val MAE = {val_mae:.4f}')

print()
print(f'=== 5-Fold CV 평균 MAE (#00 기준 점수): {np.mean(val_maes):.4f} (+/- {np.std(val_maes):.4f}) ===')


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000464 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2499
[LightGBM] [Info] Number of data points in the train set: 2395, number of used features: 32
[LightGBM] [Info] Start training from score 0.485077
fold 1: val MAE = 0.2141
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000383 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2505
[LightGBM] [Info] Number of data points in the train set: 2395, number of used features: 32
[LightGBM] [Info] Start training from score 0.482288
fold 2: val MAE = 0.2079
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000410 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2499
[LightGBM] [Info] Number of data points in the train set: 2395, number o

## 6. 최종 학습 및 예측 (전체 train 사용)

In [36]:
final_model = LGBMRegressor(random_state=RANDOM_STATE)
final_model.fit(x_train, y_train)

pred = final_model.predict(x_test)
pred[:5]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000383 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2526
[LightGBM] [Info] Number of data points in the train set: 2994, number of used features: 32
[LightGBM] [Info] Start training from score 0.482014


array([0.56732818, 0.70743812, 0.28576897, 0.47726582, 0.61712902])

## 7. Submission

결과 파일은 `data/` 폴더가 아니라 별도의 `submissions/` 폴더에 저장됩니다.

In [37]:
import os
os.makedirs('../submissions', exist_ok=True)

sample_submission['stress_score'] = pred
sample_submission.to_csv('../submissions/submit_00_baseline.csv', index=False)
sample_submission.head()

,ID,stress_score
0,TEST_0000,0.567328
1,TEST_0001,0.707438
2,TEST_0002,0.285769
3,TEST_0003,0.477266
4,TEST_0004,0.617129
